# 03 - Prepare Gold Benchmark

Converts the existing 190-question Turkish legal benchmark into the locked Q-A-Doc format used for retrieval and generation evaluation. This benchmark must not be used for fine-tuning, embedding tuning, reranker training, or synthetic training generation.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_input = DRIVE_ROOT / 'data/benchmark/turkish_legal_gold190_working_clean.csv'
corpus_path = DRIVE_ROOT / config['main_law_corpus_csv']
output_csv = DRIVE_ROOT / config['benchmark_csv']
report_json = DRIVE_ROOT / 'reports/benchmark_preparation_report_v1.json'

for path in [benchmark_input, corpus_path]:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_input, corpus_path, output_csv, report_json

In [ ]:
from src.prepare_benchmark import prepare_gold_benchmark

report = prepare_gold_benchmark(
    benchmark_input=benchmark_input,
    normalized_corpus=corpus_path,
    output_csv=output_csv,
    report_json=report_json,
)

report

In [ ]:
import pandas as pd

gold_df = pd.read_csv(output_csv, dtype=str, keep_default_na=False)
print(gold_df.shape)
print(gold_df[['question_id', 'topic', 'difficulty', 'gold_doc_keys', 'gold_article_keys', 'gold_law', 'gold_article_no']].head(10).to_string(index=False))

assert gold_df['question'].str.strip().ne('').all()
assert gold_df['gold_answer'].str.strip().ne('').all()
assert gold_df['gold_doc_keys'].str.strip().ne('').all()
assert gold_df['gold_article_keys'].str.strip().ne('').all()
print('Gold benchmark schema/content check passed.')

Expected output:

- `data/benchmark/gold_benchmark_v1.csv`
- `reports/benchmark_preparation_report_v1.json`

Next step: run retrieval evaluation using this locked benchmark. Do not use this file in any training dataset.